In [ ]:
from utils import *
import pandas as pd

# Finds the optimal $\alpha$ for the hybrid prior

In [18]:
df_gender = pd.read_csv('data/gpt-4o-mini/generated_personas_occupation_from_winogender_gender_gpt-4o-mini-2024-07-18_100_11-11-2024, 12:02:49.csv')

In [19]:
df_n = df_gender[df_gender['gender'] == 'N']
df_f = df_gender[df_gender['gender'] == 'F']
df_m = df_gender[df_gender['gender'] == 'M']

In [20]:
df_gender_sampled = pd.concat([df_n.sample(frac=0.5, random_state=1), df_f.sample(frac=0.5, random_state=1), df_m.sample(frac=0.5, random_state=1)])
print('Length df_gender_sampled:', len(df_gender))

Length df_gender_sampled: 37801


In [ ]:
df_gender_software_engineer = df_gender[df_gender['occupation'] == 'software engineer']
print('Length df_gender_software_engineer', len(df_gender_software_engineer))


Length df_gender_software_engineer 600


In [23]:
sampled_occupations = list(df_gender_sampled['occupation'].unique())
print(sampled_occupations)
print('software engineer' in sampled_occupations)

['administrator', 'paramedic', 'specialist', 'chemist', 'physician', 'investigator', 'pathologist', 'janitor', 'plumber', 'planner', 'machinist', 'pharmacist', 'electrician', 'software engineer', 'nurse', 'programmer', 'doctor', 'dispatcher', 'counselor', 'pilot', 'painter', 'technician', 'lawyer', 'receptionist', 'examiner', 'appraiser', 'dietitian', 'mechanic', 'engineer', 'librarian', 'firefighter', 'accountant', 'hairdresser', 'architect', 'cook', 'bartender', 'veterinarian', 'broker', 'paralegal', 'teacher', 'therapist', 'instructor', 'chef', 'nutritionist', 'cashier', 'practitioner', 'advisor', 'clerk', 'inspector', 'baker', 'educator', 'salesperson', 'psychologist', 'scientist', 'surgeon', 'secretary', 'worker', 'hygienist', 'carpenter', 'auditor', 'manager', 'officer', 'supervisor']
True


In [24]:
diff_alpha_vals = dict()
for i in range(21):
    alpha = i/20
    dv3_mw, dv3_mw_names, dv3_mw_by_occ, dv3_mw_by_occ_names, dv3_mw_occ, dv3_mw_occ_names = compute_calibrated_marked_words(df_gender_software_engineer, ['software engineer'], alpha,inferred_gender=False)
    diff_alpha_vals[alpha] = dv3_mw_by_occ
    print(alpha, dv3_mw_by_occ)

0.0 {'software engineer': {'F': ['her', 'she', 'women', 'sarah', 'diversity', 'maya', 'stem', 'female', 'advocate', 'girls', 'inclusive', 'chen', 'tech', 'careers', 'emily', 'field', 'young', 'technology', 'advocacy', 'in', 'award', 'inclusivity', 'empowering', 'pioneering', 'barriers', 'undergraduate', 'promoting', 'organization', 'ai', 'advocating', 'sophia', 'nonprofit', 'herself', 'representation', 'fields', 'stanford', 'masters', 'organizations', 'emma', 'inspire', 'initiative', 'soughtafter', 'trailblazer', 'inclusion', 'mit', 'pursuing', 'empathetic', 'mentorship', 'california', 'minorities', 'francisco', 'diverse', 'clara', 'san', 'forbes', 'completing', 'yoga', 'processing', 'workplace', 'research', 'prestigious', 'groundbreaking', 'workshops', 'numerous', 'dr', 'aimed', 'empower', 'achievements', 'communities', 'speaker', 'supportive', 'awards', 'practicing', 'elena', 'jessica', 'mia', 'encouraged', 'woman', 'predominantly', 'inspired', 'passionate', 'equitable', 'accessibili

In [25]:
all_words = {'F': set(), 'M': set(), 'N': set()}
for key in diff_alpha_vals.keys():
    for g in all_words.keys():
        for word in diff_alpha_vals[key]['software engineer'][g]:
            all_words[g].add(word)


In [26]:
words_shared_across_all_alpha_vals = {'F': set(), 'M': set(), 'N': set()}
for g in all_words.keys():
    for word in all_words[g]:
        in_all_alpha_vals = True
        for key in diff_alpha_vals.keys():
            if word not in diff_alpha_vals[key]['software engineer'][g]:
                in_all_alpha_vals = False
                break
        if in_all_alpha_vals:
            words_shared_across_all_alpha_vals[g].add(word)
    

In [27]:
for g in words_shared_across_all_alpha_vals.keys():
    print(g)
    for key in diff_alpha_vals.keys():
        words_not_shared = list()
        for word in diff_alpha_vals[key]['software engineer'][g]:
            if word not in words_shared_across_all_alpha_vals[g]:
                words_not_shared.append(word)
        print(key, len(words_not_shared), sorted(words_not_shared))
    print('---------------------')

F
0.0 71 ['academic', 'accessibility', 'accolades', 'actively', 'advancing', 'aidriven', 'aisha', 'algorithms', 'collaborates', 'conferences', 'continue', 'contributions', 'countless', 'cum', 'demonstrating', 'earning', 'educators', 'empowered', 'equitable', 'equity', 'everyone', 'extends', 'faced', 'featured', 'focused', 'focuses', 'fostering', 'future', 'gap', 'health', 'hiring', 'immigrants', 'imposter', 'industry', 'innovators', 'laude', 'leadership', 'massachusetts', 'mental', 'navigating', 'networking', 'nonprofits', 'others', 'panels', 'passionate', 'pave', 'perseverance', 'perspectives', 'promote', 'proving', 'prowess', 'publications', 'recipes', 'recognition', 'recognized', 'recognizing', 'rescue', 'resilience', 'resources', 'stereotypes', 'summa', 'supportive', 'syndrome', 'tensorflow', 'traditionally', 'trailblazing', 'underrepresented', 'unwavering', 'vocal', 'volunteering', 'workplace']
0.05 69 ['academic', 'accessibility', 'accolades', 'actively', 'advancing', 'aidriven',

In [28]:
# table for paper
for g in words_shared_across_all_alpha_vals.keys():
    print(g)
    print('\\begin{longtable}{@{}p{0.2\\textwidth}@{}p{0.75\\textwidth}}')
    print('\\toprule')
    print('\\textbf{$p$} & Words Not Shared\\\\')
    print('\\midrule')
    print('\\endfirsthead')

    print('\\toprule')
    print('\\textbf{$p$} & Words Not Shared\\\\')
    print('\\midrule')
    print('\\endhead')

    print('\\midrule')
    print('\\multicolumn{2}{r}{\\textit{Continued on next page}}\\\\')
    print('\\midrule')
    print('\\endfoot')

    print('\\bottomrule')
    print('\\endlastfoot')

    
    for key in diff_alpha_vals.keys():
        words_to_print = ""
        words_not_shared = list()
        for word in diff_alpha_vals[key]['software engineer'][g]:
            if word not in words_shared_across_all_alpha_vals[g]:
                words_not_shared.append(word)
        for word in sorted(words_not_shared):
            words_to_print += f"{word}, "
        print(f"{key} & {words_to_print[:-2]}\\\\")
    print('\\bottomrule')
    print('\\end{longtable}')

F
\begin{longtable}{@{}p{0.2\textwidth}@{}p{0.75\textwidth}}
\toprule
\textbf{$p$} & Words Not Shared\\
\midrule
\endfirsthead
\toprule
\textbf{$p$} & Words Not Shared\\
\midrule
\endhead
\midrule
\multicolumn{2}{r}{\textit{Continued on next page}}\\
\midrule
\endfoot
\bottomrule
\endlastfoot
0.0 & academic, accessibility, accolades, actively, advancing, aidriven, aisha, algorithms, collaborates, conferences, continue, contributions, countless, cum, demonstrating, earning, educators, empowered, equitable, equity, everyone, extends, faced, featured, focused, focuses, fostering, future, gap, health, hiring, immigrants, imposter, industry, innovators, laude, leadership, massachusetts, mental, navigating, networking, nonprofits, others, panels, passionate, pave, perseverance, perspectives, promote, proving, prowess, publications, recipes, recognition, recognized, recognizing, rescue, resilience, resources, stereotypes, summa, supportive, syndrome, tensorflow, traditionally, trailblazing, u

In [29]:
# markdown table for github repo
for g in words_shared_across_all_alpha_vals.keys():
    print(g)
    print("|$\\alpha$|Words Not Shared|")
    print("|-|-|")
    for key in diff_alpha_vals.keys():
        words_to_print = ""
        words_not_shared = list()
        for word in diff_alpha_vals[key]['software engineer'][g]:
            if word not in words_shared_across_all_alpha_vals[g]:
                words_not_shared.append(word)
        for word in sorted(words_not_shared):
            words_to_print += f"{word}, "
        print(f"|{key} | {words_to_print[:-2]}|")
    print('-------------------')

F
|$\alpha$|Words Not Shared|
|-|-|
|0.0 | academic, accessibility, accolades, actively, advancing, aidriven, aisha, algorithms, collaborates, conferences, continue, contributions, countless, cum, demonstrating, earning, educators, empowered, equitable, equity, everyone, extends, faced, featured, focused, focuses, fostering, future, gap, health, hiring, immigrants, imposter, industry, innovators, laude, leadership, massachusetts, mental, navigating, networking, nonprofits, others, panels, passionate, pave, perseverance, perspectives, promote, proving, prowess, publications, recipes, recognition, recognized, recognizing, rescue, resilience, resources, stereotypes, summa, supportive, syndrome, tensorflow, traditionally, trailblazing, underrepresented, unwavering, vocal, volunteering, workplace|
|0.05 | academic, accessibility, accolades, actively, advancing, aidriven, algorithms, collaborates, conferences, continue, contributions, countless, cum, demonstrating, earning, educators, empowe

# Words Not Shared across different values of $\alpha$ for sampled software engineers
|$\alpha$|Words Not Shared|
|-|-|
|0.0 | academic, accessibility, accolades, actively, advancing, aidriven, aisha, algorithms, collaborates, conferences, continue, contributions, countless, cum, demonstrating, earning, educators, empowered, equitable, equity, everyone, extends, faced, featured, focused, focuses, fostering, future, gap, health, hiring, immigrants, imposter, industry, innovators, laude, leadership, massachusetts, mental, navigating, networking, nonprofits, others, panels, passionate, pave, perseverance, perspectives, promote, proving, prowess, publications, recipes, recognition, recognized, recognizing, rescue, resilience, resources, stereotypes, summa, supportive, syndrome, tensorflow, traditionally, trailblazing, underrepresented, unwavering, vocal, volunteering, workplace|
|0.05 | academic, accessibility, accolades, actively, advancing, aidriven, algorithms, collaborates, conferences, continue, contributions, countless, cum, demonstrating, earning, educators, empowered, equitable, equity, everyone, extends, faced, featured, focused, focuses, fostering, future, gap, health, hiring, immigrants, imposter, industry, innovators, language, laude, leadership, massachusetts, mental, navigating, networking, nonprofits, others, panels, passionate, pave, perseverance, promote, proving, prowess, publications, recipes, recognition, recognized, recognizing, rescue, resilience, resources, stereotypes, summa, supportive, syndrome, tensorflow, traditionally, trailblazing, underrepresented, unwavering, volunteering, workplace|
|0.1 | academic, accessibility, accolades, actively, advancing, aidriven, algorithms, collaborates, conferences, continue, contributions, countless, cum, demonstrating, earning, educators, empowered, equitable, equity, everyone, extends, faced, featured, focused, focuses, fostering, future, gap, health, hiring, immigrants, imposter, industry, innovators, language, laude, leadership, massachusetts, mental, navigating, networking, nonprofits, others, panels, passionate, pave, perseverance, promote, proving, prowess, publications, recipes, recognition, recognized, rescue, resilience, resources, stereotypes, summa, supportive, syndrome, tensorflow, traditionally, trailblazing, unwavering, volunteering, workplace|
|0.15 | academic, accessibility, accolades, actively, advancing, aidriven, algorithms, collaborates, conferences, continue, contributions, countless, cum, demonstrating, doctorate, earning, educators, ellie, empowered, equitable, equity, everyone, extends, faced, featured, focused, focuses, fostering, future, gap, health, hiring, immigrants, imposter, industry, innovators, language, laude, leadership, massachusetts, mental, navigating, networking, nonprofits, others, panels, passionate, pave, perseverance, promote, proving, prowess, publications, recipes, recognition, recognized, rescue, resilience, resources, stereotypes, summa, supportive, syndrome, tensorflow, traditionally, trailblazing, unwavering, volunteering, workplace, workplaces|
|0.2 | academic, accessibility, accolades, actively, advancing, aidriven, aisha, algorithms, collaborates, conferences, continue, contributions, countless, cum, demonstrating, disparity, doctorate, earning, educators, ellie, empowered, equitable, equity, everyone, extends, faced, featured, focused, focuses, fostering, future, gap, health, immigrants, imposter, industry, innovators, language, laude, massachusetts, mental, navigating, networking, nonprofits, others, panels, passionate, perseverance, promote, proving, prowess, publications, recognition, recognized, rescue, resilience, resources, stereotypes, summa, supportive, syndrome, tensorflow, traditionally, trailblazing, tran, unwavering, volunteering, workforce, workplace, workplaces|
|0.25 | academic, accessibility, advancing, aidriven, aisha, algorithms, collaborates, conferences, continue, contributions, countless, cum, disparity, doctorate, earning, educators, ellie, empowered, equitable, equity, everyone, extends, faced, featured, focused, focuses, fostering, future, gap, health, immigrants, imposter, industry, innovators, language, laude, massachusetts, mental, navigating, networking, nguyen, others, panels, passionate, perseverance, priya, promote, proving, prowess, publications, recognition, recognized, rescue, resilience, resources, stereotypes, summa, supportive, syndrome, tensorflow, traditionally, trailblazing, tran, underrepresentation, volunteering, workforce, workplace, workplaces|
|0.3 | academic, accessibility, advancing, aidriven, aisha, algorithms, collaborates, conferences, continue, contributions, countless, cum, disparity, doctorate, educators, ellie, empowered, equitable, equity, everyone, extends, faced, featured, focused, focuses, fostering, gap, health, immigrants, imposter, industry, innovators, language, laude, massachusetts, mental, navigating, networking, nguyen, others, panels, passionate, perseverance, priya, promote, proving, publications, recognition, recognized, rescue, resilience, resources, stereotypes, summa, supportive, syndrome, traditionally, trailblazing, tran, underrepresentation, volunteering, workforce, workplace|
|0.35 | academic, accessibility, aidriven, aisha, algorithms, collaborates, conferences, continue, contributions, countless, cum, disparity, doctorate, educators, ellie, empowered, equitable, equity, everyone, extends, faced, featured, focused, focuses, fostering, gap, immigrants, imposter, industry, innovators, language, laude, massachusetts, mental, navigating, networking, nguyen, others, panels, passionate, perseverance, priya, promote, proving, publications, recognition, recognized, rescue, resilience, resources, stereotypes, summa, supportive, syndrome, traditionally, trailblazing, tran, underrepresentation, volunteering, workforce, workplace|
|0.4 | academic, accessibility, aidriven, aisha, algorithms, collaborates, conferences, continue, contributions, countless, cum, disparity, doctorate, educators, ellie, empowered, entering, equitable, everyone, extends, faced, featured, focused, focuses, fostering, gap, immigrants, imposter, industry, language, laude, massachusetts, mental, navigating, networking, nguyen, others, panels, passionate, perseverance, priya, promote, proving, publications, recognition, recognized, rescue, resilience, resources, stereotypes, summa, supportive, syndrome, traditionally, trailblazing, tran, underrepresentation, volunteering, workforce, workplace|
|0.45 | academic, accessibility, aidriven, aisha, algorithms, anitaborg, collaborates, conferences, continue, contributions, countless, cum, disparity, doctorate, educators, ellie, empowered, equitable, everyone, extends, faced, featured, focused, focuses, fostering, gap, immigrants, imposter, industry, language, laude, massachusetts, mental, navigating, networking, nguyen, others, panels, passionate, perseverance, priya, promote, proving, publications, recognized, rescue, resilience, resources, rodriguez, stereotypes, summa, supportive, syndrome, traditionally, trailblazing, tran, underrepresentation, volunteering, workforce, workplace|
|0.5 | academic, accessibility, aidriven, aisha, algorithms, anitaborg, collaborates, conferences, continue, contributions, countless, cum, disparity, doctorate, educators, ellie, empowered, equitable, everyone, extends, faced, featured, focused, focuses, fostering, gap, immigrants, imposter, language, laude, massachusetts, mental, navigating, networking, nguyen, others, panels, passionate, priya, promote, proving, publications, published, recognized, rescue, resilience, resources, rodriguez, stereotypes, summa, supportive, syndrome, trailblazing, tran, underrepresentation, volunteering, workforce, workplace|
|0.55 | academic, accessibility, aidriven, aisha, algorithms, anitaborg, collaborates, conferences, continue, contributions, countless, cum, disparity, doctorate, educators, ellie, empowered, equitable, everyone, extends, faced, featured, focused, focuses, fostering, gap, immigrants, imposter, language, laude, massachusetts, mental, navigating, networking, nguyen, others, panels, passionate, priya, promote, proving, publications, published, recognized, rescue, resilience, resources, rodriguez, stereotypes, summa, supportive, syndrome, trailblazing, tran, underrepresentation, volunteering, workforce, workplace|
|0.6 | academic, accessibility, aidriven, aisha, algorithms, anitaborg, collaborates, conferences, continue, contributions, countless, cum, doctorate, educators, ellie, empowered, equitable, everyone, extends, faced, featured, focused, focuses, fostering, gap, immigrants, imposter, language, laude, massachusetts, mental, navigating, networking, nguyen, others, panels, passionate, priya, promote, proving, publications, published, recognized, resilience, resources, rodriguez, stereotypes, summa, supportive, syndrome, trailblazing, tran, underrepresentation, volunteering, workforce, workplace|
|0.65 | academic, accessibility, aidriven, aisha, algorithms, anitaborg, collaborates, conferences, continue, contributions, countless, cum, doctorate, educators, ellie, equitable, everyone, faced, featured, focused, focuses, fostering, gap, immigrants, imposter, language, laude, massachusetts, navigating, networking, nguyen, others, overcome, panels, passionate, priya, promote, proving, publications, published, recognized, resilience, resources, rodriguez, stereotypes, summa, supportive, syndrome, trailblazing, tran, underrepresentation, volunteering, workforce, workplace|
|0.7 | aidriven, aisha, algorithms, anitaborg, collaborates, conferences, continue, contributions, countless, cum, doctorate, educators, ellie, equitable, everyone, faced, featured, focused, fostering, gap, immigrants, imposter, language, laude, massachusetts, navigating, networking, nguyen, others, panels, passionate, priya, proving, publications, published, recognized, resilience, resources, rodriguez, summa, supportive, syndrome, trailblazing, tran, underrepresentation, volunteering, workforce, workplace|
|0.75 | aidriven, aisha, conferences, continue, contributions, countless, cum, doctorate, educators, ellie, equitable, everyone, faced, featured, focused, gap, immigrants, imposter, kitchen, language, laude, massachusetts, nguyen, others, panels, passionate, priya, proving, publications, published, recognized, resilience, resources, supportive, trailblazing, tran, underrepresentation, volunteering, workplace|
|0.8 | aidriven, aisha, conferences, continue, cum, doctorate, educators, ellie, equitable, everyone, faced, featured, focused, gap, kitchen, language, laude, massachusetts, nguyen, panels, passionate, priya, proving, publications, published, recognized, resilience, resources, supportive, trailblazing, underrepresentation, volunteering, workplace|
|0.85 | aidriven, aisha, conferences, continue, doctorate, ellie, equitable, everyone, featured, focused, gap, kitchen, language, massachusetts, nguyen, panels, passionate, priya, publications, published, recognized, resilience, resources, supportive, trailblazing, workplace|
|0.9 | aisha, being, conferences, ellie, equitable, everyone, featured, focused, gap, kitchen, language, panels, passionate, publications, published, supportive, trailblazing, workplace|
|0.95 | aisha, being, conferences, ellie, equitable, featured, kitchen, language, one, passionate, published|
|1.0 | aisha, been, being, i, kitchen, language, one, program, published|


# Words not shared across different values of $\alpha$ for sample non-binary software engineers
|$\alpha$|Words Not Shared|
|-|-|
|0.0 | actively, activism, aimed, align, authentically, background, beacon, blending, blossomed, break, broader, championed, coastal, creativity, css, culture, disabilities, discussions, ecofriendly, educate, embraces, empathy, empowered, environment, express, faced, faces, fields, focused, focuses, frontend, galleries, generations, influenced, initiatives, inspire, journey, listener, multicultural, nonprofit, oneself, outspoken, particularly, passionate, pave, pioneering, practicing, promotes, proving, pursuing, resonate, selfcare, shaped, societal, speculative, stereotypes, striving, tapestry, themes, transcends, uiux, uplift, usable, user, usercentered, valued, vibrant, wellbeing, worlds|
|0.05 | actively, activism, aimed, align, authentically, background, beacon, blending, blossomed, break, broader, casey, championed, coastal, creativity, css, culture, disabilities, discussions, ecofriendly, educate, embraces, empathy, empowered, environment, express, faced, faces, fields, focused, focuses, frontend, galleries, generations, influenced, initiatives, inspire, installations, journey, listener, maledominated, multicultural, nonprofit, oneself, outspoken, particularly, passionate, pave, pioneering, practicing, promotes, proving, pursuing, resonate, selfcare, shaped, societal, speculative, stereotypes, striving, tapestry, themes, transcends, uiux, uplift, usable, user, usercentered, valued, vibrant, wellbeing, worlds|
|0.1 | activism, aimed, align, authentically, background, beacon, blending, blossomed, break, broader, casey, championed, coastal, creativity, css, culture, disabilities, discussions, ecofriendly, educate, embraces, empathy, empowered, environment, express, faced, faces, fields, focused, focuses, frontend, galleries, generations, influenced, inspire, installations, listener, maledominated, multicultural, nonprofit, oneself, outspoken, particularly, passionate, pave, pioneering, practicing, promotes, proving, pursuing, resonate, selfcare, shaped, societal, speculative, stereotypes, striving, tapestry, themes, transcends, uiux, uplift, usable, user, usercentered, vibrant, wellbeing, worlds|
|0.15 | activism, aimed, align, authentically, background, beacon, blending, blossomed, break, broader, casey, championed, coastal, creativity, css, culture, disabilities, discussions, ecofriendly, educate, embraces, empathy, empowered, environment, establish, express, faced, faces, fields, focused, focuses, frontend, galleries, generations, influenced, inspire, installations, listener, maledominated, more, multicultural, nonprofit, oneself, outspoken, particularly, passionate, pave, pioneering, practicing, promotes, proving, pursuing, resonate, selfcare, shaped, societal, speculative, stereotypes, tapestry, themes, transcends, uiux, uplift, usable, user, usercentered, vibrant, wellbeing, worlds|
|0.2 | activism, aimed, align, authentically, background, beacon, blending, blossomed, break, broader, casey, championed, coastal, creativity, css, culture, disabilities, discussions, ecofriendly, educate, embraces, empathy, empowered, environment, establish, express, faced, faces, fields, focused, focuses, frontend, galleries, generations, influenced, inspire, installations, listener, maledominated, more, multicultural, nonprofit, oneself, outspoken, particularly, passionate, pave, pioneering, practicing, promotes, proving, pursuing, resonate, selfcare, shaped, societal, speculative, stereotypes, tapestry, themes, transcends, uiux, uplift, usable, user, usercentered, vibrant, wellbeing, worlds|
|0.25 | activism, aimed, align, authentically, background, beacon, blending, blossomed, break, broader, casey, championed, coastal, creativity, css, culture, disabilities, discussions, ecofriendly, educate, embraces, empathy, empowered, environment, establish, express, faced, faces, fields, focused, focuses, frontend, galleries, generations, influenced, inspire, installations, listener, maledominated, more, multicultural, nonprofit, oneself, outspoken, particularly, passionate, pave, pioneering, practicing, promotes, proving, pursuing, quinn, related, resonate, shaped, societal, speculative, stereotypes, tapestry, themes, transcends, uiux, uplift, usable, user, usercentered, vibrant, wellbeing, worlds|
|0.3 | activism, aimed, align, authentically, background, beacon, blending, blossomed, break, broader, casey, championed, coastal, creativity, culture, disabilities, discussions, ecofriendly, educate, embraces, empathy, empowered, environment, establish, express, faced, faces, fields, focused, focuses, frontend, galleries, generations, influenced, inspire, installations, listener, maledominated, more, multicultural, nonprofit, oneself, outspoken, particularly, passionate, pave, pioneering, practicing, promotes, proving, pursuing, quinn, related, resonate, shaped, societal, speculative, stereotypes, tapestry, themes, transcends, uiux, uplift, usable, user, usercentered, vibrant, wellbeing, worlds|
|0.35 | activism, aimed, align, authentically, background, beacon, blending, blossomed, break, broader, casey, championed, coastal, creativity, culture, disabilities, discussions, ecofriendly, educate, embraces, empathy, empowered, environment, establish, express, faced, faces, fields, focused, focuses, frontend, galleries, generations, influenced, inspire, installations, listener, maledominated, more, multicultural, nonprofit, oneself, outspoken, particularly, passionate, pave, pioneering, promotes, proving, pursuing, quinn, related, resonate, shaped, societal, speculative, stereotypes, tapestry, themes, transcends, uiux, uplift, usable, user, usercentered, vibrant, wellbeing, worlds|
|0.4 | activism, aimed, align, authentically, beacon, blending, blossomed, break, broader, casey, championed, coastal, creativity, culture, disabilities, discussions, ecofriendly, educate, embraces, empathy, empowered, environment, establish, express, faced, faces, fields, focused, focuses, frontend, galleries, generations, influenced, inspire, installations, maledominated, more, multicultural, nonprofit, oneself, outspoken, particularly, passionate, pioneering, promotes, proving, pursuing, quinn, related, resonate, shaped, societal, speculative, stereotypes, tapestry, themes, transcends, uiux, uplift, usable, user, usercentered, vibrant, wellbeing, worlds|
|0.45 | activism, aimed, align, authentically, beacon, blending, blossomed, break, broader, casey, championed, coastal, creativity, culture, disabilities, discussions, ecofriendly, educate, embraces, empathy, empowered, environment, establish, express, faced, faces, fields, focused, focuses, frontend, galleries, genderdiverse, genderneutral, generations, influenced, inspire, installations, intersectionality, microaggressions, more, multicultural, nonprofit, oneself, outspoken, particularly, passionate, pioneering, promotes, proving, pursuing, quinn, related, resonate, selfacceptance, shaped, societal, speculative, stereotypes, tapestry, themes, transcends, uiux, uplift, usable, user, usercentered, vibrant, wellbeing, worlds|
|0.5 | activism, aimed, align, authentically, beacon, blending, blossomed, break, broader, casey, championed, coastal, creativity, culture, disabilities, discussions, ecofriendly, educate, empathy, empowered, environment, establish, express, expression, faced, faces, fields, focused, focuses, frontend, galleries, genderdiverse, genderneutral, generations, influenced, inspire, installations, intersectionality, microaggressions, more, multicultural, needs, nonprofit, oneself, outspoken, passionate, pioneering, proving, pursuing, quinn, related, resonate, selfacceptance, shaped, societal, speculative, stereotypes, tapestry, themes, transcends, uiux, uplift, usable, usercentered, vibrant, wellbeing, worlds|
|0.55 | activism, aimed, align, authentically, beacon, blending, blossomed, break, broader, casey, championed, coastal, creativity, culture, disabilities, discussions, ecofriendly, educate, empathy, empowered, environment, establish, express, expression, faced, faces, fields, focused, focuses, frontend, galleries, genderdiverse, genderneutral, generations, influenced, inspire, installations, intersectionality, microaggressions, more, multicultural, needs, nonprofit, oneself, outspoken, passionate, pioneering, proving, pursuing, quinn, related, resonate, selfacceptance, shaped, societal, speculative, stereotypes, tapestry, themes, transcends, uiux, uplift, usable, usercentered, value, vibrant, worlds|
|0.6 | activism, aimed, authentically, beacon, blending, blossomed, break, broader, casey, championed, coastal, creativity, culture, disabilities, discussions, ecofriendly, educate, empathy, empowered, environment, establish, express, expression, faced, faces, fields, focused, focuses, frontend, galleries, influenced, inspire, installations, more, multicultural, needs, nonprofit, oneself, outspoken, passionate, pioneering, pursuing, quinn, related, resonate, shaped, societal, speculative, stereotypes, tapestry, themes, transcends, uiux, uplift, usable, usercentered, value, worlds|
|0.65 | activism, aimed, authentically, beacon, beliefs, blending, blossomed, break, broader, casey, championed, coastal, creativity, culture, disabilities, discussions, ecofriendly, educate, empathy, empowered, environment, establish, express, expression, faced, faces, fields, focused, focuses, influenced, inspire, installations, more, multicultural, needs, nonprofit, oneself, outspoken, passionate, pursuing, quinn, related, societal, speculative, stereotypes, tapestry, themes, transcends, uiux, uplift, usable, usercentered, value, worlds|
|0.7 | activism, aimed, authentically, blending, blossomed, break, broader, casey, championed, coastal, culture, disabilities, discussions, ecofriendly, educate, empathy, environment, establish, express, expression, faced, faces, fields, focused, focuses, influenced, inspire, more, multicultural, needs, nonprofit, oneself, outspoken, passionate, pursuing, quinn, related, societal, stereotypes, tapestry, themes, uiux, uplift, usercentered, value|
|0.75 | activism, aimed, authentically, blending, blossomed, break, broader, casey, championed, coastal, culture, discussions, ecofriendly, educate, empathy, environment, establish, express, expression, faced, faces, fields, focused, focuses, influenced, inspire, more, multicultural, needs, nonprofit, oneself, outspoken, passionate, pursuing, quinn, related, societal, stereotypes, tapestry, themes, uiux, value|
|0.8 | activism, aimed, authentically, blending, casey, championed, coastal, culture, discussions, ecofriendly, educate, empathy, environment, establish, express, expression, faced, faces, feel, fields, focused, focuses, influenced, inspire, more, needs, nonprofit, outspoken, passionate, pursuing, quinn, related, societal, themes, uiux, value, welcome|
|0.85 | activism, blending, casey, championed, coastal, culture, discussions, ecofriendly, educate, empathy, environment, establish, express, expression, faced, faces, feel, fields, focused, focuses, influenced, inspire, more, needs, nonprofit, outspoken, passionate, pursuing, quinn, related, societal, themes, uiux, value, welcome|
|0.9 | activism, blending, casey, championed, discussions, educate, empathy, establish, express, expression, faced, feel, focused, influenced, inspire, more, nonprofit, outspoken, passionate, pursuing, quinn, should, societal, state, themes, uiux, value, welcome|
|0.95 | activism, casey, discussions, educate, establish, express, expression, faced, feel, felt, inspire, more, passionate, pursuing, should, societal, state, themes, uiux, value, we, welcome, were|
|1.0 | be, establish, expression, face, feel, felt, live, more, our, should, state, strength, value, we, welcome, were|
